# OpenMonitor AIA ROI light-curve notebook

**EN.** Use this notebook to read an AIA ROI result exported by OpenMonitor Solar Lab. You can paste a result id, a job id, or a Solar Lab share link, or upload the downloaded files manually.

**ES.** Usa este cuaderno para leer un resultado AIA ROI exportado por OpenMonitor Solar Lab. Puedes pegar un result id, un job id o un enlace compartido, o subir los archivos manualmente.

**PT.** Use este notebook para ler um resultado AIA ROI exportado pelo OpenMonitor Solar Lab. Voce pode colar um result id, job id ou link compartilhado, ou enviar os arquivos manualmente.

Recommended files from the web tool: `lightcurves.fits` plus `metadata.json`. CSV/ECSV also work. Movies and PNGs are optional visual context.


## 1. Runtime setup

This installs only common scientific Python packages if Colab does not already provide them.


In [ ]:
import importlib.util
import subprocess
import sys

packages = ["astropy", "pandas", "matplotlib", "numpy", "requests"]
missing = [pkg for pkg in packages if importlib.util.find_spec(pkg) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

print("Runtime ready.")


## 2. Choose input

This notebook is preloaded with a public demo run from Solar OpenMonitor:

- Recovery code: `aia_963db1ee7c6b49b9a9`
- Event: X1.6 flare, 2014-10-22
- AIA channel: 1700 Å
- Cadence: 24 s

Run all cells to download the cached OpenMonitor result and inspect the FITS/CSV/metadata exports.

You can replace `OPENMONITOR_RESULT` with any of these:

- a Solar Lab share link containing `aiaJob=...`
- an AIA recovery code such as `aia_abc123...`
- a result id/cache key such as `b03c1a0b004f04f0d0b8`
- an API result URL such as `https://api.openmonitor.org/api/solar/aia/results/<id>.json`

Leave `OPENMONITOR_RESULT` empty if you prefer manual upload.


In [ ]:
OPENMONITOR_RESULT = "aia_963db1ee7c6b49b9a9"  # paste a result id, job id, or share link here
API_BASE = "https://api.openmonitor.org"
WORKDIR = "openmonitor_aia_result"

# If OPENMONITOR_RESULT is empty in Colab, the next cells will ask you to upload files.


## 3. Fetch from OpenMonitor or upload files

The notebook downloads only processed result artifacts. It does not fetch the private/raw server-side JSOC cache.


In [ ]:
from pathlib import Path
from urllib.parse import parse_qs, urlparse
import json
import re
import requests

def parse_openmonitor_reference(value):
    value = str(value or "").strip()
    if not value:
        return None, None
    parsed = urlparse(value)
    if parsed.scheme and parsed.netloc:
        query = parse_qs(parsed.query)
        if query.get("aiaJob"):
            return "job", query["aiaJob"][0]
        parts = [part for part in parsed.path.split("/") if part]
        if "results" in parts:
            token = parts[parts.index("results") + 1]
            return "result", token.replace(".json", "")
    parts = [part for part in parsed.path.split("/") if part]
    if "results" in parts:
        token = parts[parts.index("results") + 1]
        return "result", token.replace(".json", "")
    if value.startswith("aia_"):
        return "job", value
    return "result", value.replace(".json", "")

def result_id_from_job(job_id):
    url = f"{API_BASE}/api/solar/aia/jobs/{job_id}"
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    job = response.json()
    if job.get("status") != "complete":
        raise RuntimeError(f"Job is not complete yet: {job.get('status')} - {job.get('errorMessage')}")
    result_url = job.get("resultUrl") or ""
    kind, token = parse_openmonitor_reference(result_url)
    if kind != "result" or not token:
        raise RuntimeError(f"Could not resolve result id from job response: {job}")
    return token

def download_url(url, path, timeout=90):
    path = Path(path)
    response = requests.get(url, timeout=timeout)
    if response.status_code == 404:
        return False
    response.raise_for_status()
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(response.content)
    return True

def artifact_url(result_id, relative_path=None):
    if relative_path is None:
        return f"{API_BASE}/api/solar/aia/results/{result_id}.json"
    return f"{API_BASE}/api/solar/aia/results/{result_id}/{relative_path}"

def fetch_openmonitor_result(reference, outdir=WORKDIR):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    kind, token = parse_openmonitor_reference(reference)
    if kind == "job":
        token = result_id_from_job(token)
    if kind is None or not token:
        raise ValueError("Paste an OpenMonitor result id, job id, API URL, or share link.")

    downloaded = []
    if download_url(artifact_url(token), outdir / "result.json"):
        downloaded.append("result.json")

    candidates = [
        "lightcurves.fits", "lightcurves.ecsv", "lightcurves.csv", "metadata.json",
        "lightcurves.png", "roi_overlay.png", "movie.gif", "manifest.json",
        "science_bundle_manifest.json", "frames_manifest.json", "request.json",
        "roi_movies/ROI_A.gif", "roi_movies/ROI_B.gif", "roi_movies/ROI_C.gif", "roi_movies/ROI_D.gif",
    ]
    for rel in candidates:
        if download_url(artifact_url(token, rel), outdir / rel):
            downloaded.append(rel)

    print(f"Resolved result id: {token}")
    print(f"Downloaded {len(downloaded)} files to {outdir.resolve()}")
    for name in downloaded:
        print(" -", name)
    return outdir

def upload_openmonitor_files(outdir=WORKDIR):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    try:
        from google.colab import files
    except Exception as exc:
        raise RuntimeError("Manual upload is available in Google Colab. Otherwise place files in WORKDIR.") from exc
    uploaded = files.upload()
    for name, data in uploaded.items():
        path = outdir / name
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(data)
    print(f"Uploaded {len(uploaded)} files to {outdir.resolve()}")
    return outdir

if OPENMONITOR_RESULT.strip():
    result_dir = fetch_openmonitor_result(OPENMONITOR_RESULT, WORKDIR)
else:
    result_dir = upload_openmonitor_files(WORKDIR)


## 4. Read the table

Priority is FITS, then ECSV, then CSV. FITS/ECSV preserve richer metadata; CSV is useful for spreadsheets.


In [ ]:
from astropy.table import Table
from IPython.display import display
import numpy as np
import pandas as pd

def first_existing(directory, names):
    directory = Path(directory)
    for name in names:
        path = directory / name
        if path.exists():
            return path
    return None

table_path = first_existing(result_dir, ["lightcurves.fits", "lightcurves.ecsv", "lightcurves.csv"])
if table_path is None:
    raise FileNotFoundError("No lightcurves.fits, lightcurves.ecsv, or lightcurves.csv found.")

table_metadata = {}
if table_path.suffix.lower() in {".fits", ".fit", ".ecsv"}:
    table = Table.read(table_path)
    table_metadata = dict(table.meta)
    df = table.to_pandas()
else:
    df = pd.read_csv(table_path)

metadata = dict(table_metadata)
metadata_path = Path(result_dir) / "metadata.json"
if metadata_path.exists():
    with metadata_path.open("r", encoding="utf-8") as handle:
        metadata.update(json.load(handle))

if "UTC" not in df.columns:
    raise ValueError("Expected a UTC column in the light-curve table.")
df["UTC"] = pd.to_datetime(df["UTC"].astype(str), utc=True)

SYSTEM_COLUMNS = {"UTC", "EXPTIME", "QUALITY"}
roi_columns = [
    col for col in df.columns
    if col not in SYSTEM_COLUMNS and not col.endswith("_sat_frac") and not col.endswith("_saturated")
]

print(f"Loaded: {table_path}")
print(f"Rows: {len(df)}")
print(f"ROI curve columns: {roi_columns}")
display(df.head())


## 5. Inspect scientific metadata

OpenMonitor exports ROI coordinates in image-plane helioprojective arcsec, not heliographic latitude/longitude. The table is intended for quick-look and relative analysis unless you apply your own AIA calibration/degradation corrections.


In [ ]:
def get_nested(data, *keys, default=None):
    cur = data
    for key in keys:
        if not isinstance(cur, dict) or key not in cur:
            return default
        cur = cur[key]
    return cur

source = metadata.get("source", {}) if isinstance(metadata.get("source"), dict) else {}
calibration = metadata.get("calibration", {}) if isinstance(metadata.get("calibration"), dict) else {}
quality = metadata.get("quality", {}) if isinstance(metadata.get("quality"), dict) else {}
pixel_scale = metadata.get("pixel_scale", {}) if isinstance(metadata.get("pixel_scale"), dict) else {}
processing = metadata.get("processing", {}) if isinstance(metadata.get("processing"), dict) else {}
roi_meta = metadata.get("roi", {}) if isinstance(metadata.get("roi"), dict) else {}

print("Source")
print("  instrument:", source.get("instrument", "SDO/AIA"))
print("  series:", source.get("series"))
print("  wavelength_angstrom:", source.get("wavelength_angstrom"))
print("  cadence_seconds:", source.get("cadence_seconds"))
print("  UTC span:", df["UTC"].iloc[0], "->", df["UTC"].iloc[-1])

print("\nProcessing")
print("  signal_mode:", processing.get("signal_mode"))
print("  normalize_exposure:", processing.get("normalize_exposure"))
print("  background:", processing.get("background"))

print("\nCalibration state")
for key in ["DATALEVEL", "AIAPREP", "PSFDECON", "DEGRADCORR", "EXPNORM"]:
    print(f"  {key}:", calibration.get(key))
if calibration.get("calibration_note"):
    print("  note:", calibration.get("calibration_note"))

print("\nFrame quality")
print("  EXPTIME column present:", "EXPTIME" in df.columns)
print("  QUALITY column present:", "QUALITY" in df.columns)
print("  quality summary:", quality)
if metadata.get("quality_note"):
    print("  note:", metadata.get("quality_note"))

print("\nPixel scale")
print(pixel_scale)

print("\nROI boxes")
for box in roi_meta.get("boxes", []):
    print(f"  {box.get('label') or box.get('id')}: center={box.get('center_arcsec')} size={box.get('size_arcsec')} bounds={box.get('bounds_arcsec')} pixels={box.get('pixel_count')}")

attribution = metadata.get("attribution", {}) if isinstance(metadata.get("attribution"), dict) else {}
if attribution:
    print("\nAttribution")
    print(attribution.get("ready_to_paste"))


## 6. Plot ROI light curves

Default plot uses the exported values exactly as provided. If `EXPNORM=true`, the values are exposure-normalized intensities, approximately DN/s. If the selected signal was flare excess, the baseline was already subtracted by OpenMonitor.


In [ ]:
import datetime as dt
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

NORMALIZE_TO_PEAK = False      # True: each ROI divided by its own max absolute value
SUBTRACT_FIRST_N_MINUTES = 0   # extra local subtraction; keep 0 to use exported values as-is
OUTPUT_PLOT = "openmonitor_aia_roi_lightcurve.png"

if not roi_columns:
    raise ValueError("No ROI curve columns found.")

plot_df = df.copy()
time = plot_df["UTC"]
fig, ax = plt.subplots(figsize=(9.5, 4.8))
colors = plt.get_cmap("tab10")

for idx, col in enumerate(roi_columns):
    y = pd.to_numeric(plot_df[col], errors="coerce").astype(float)
    if SUBTRACT_FIRST_N_MINUTES and SUBTRACT_FIRST_N_MINUTES > 0:
        t0 = time.iloc[0]
        mask = time <= t0 + pd.Timedelta(minutes=SUBTRACT_FIRST_N_MINUTES)
        y = y - y[mask].mean()
    if NORMALIZE_TO_PEAK:
        peak = np.nanmax(np.abs(y))
        if peak and np.isfinite(peak):
            y = y / peak
    ax.plot(time, y, linewidth=1.8, label=col, color=colors(idx % 10))

unit = "normalized" if NORMALIZE_TO_PEAK else ("DN/s" if calibration.get("EXPNORM") else "DN or exported units")
ax.set_xlabel("Time (UTC)")
ax.set_ylabel(f"ROI mean intensity ({unit})")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M", tz=dt.timezone.utc))
ax.legend(frameon=False, loc="best")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.autofmt_xdate(rotation=0, ha="center")
fig.tight_layout()
fig.savefig(OUTPUT_PLOT, dpi=180, bbox_inches="tight")
plt.show()
print("Saved plot:", OUTPUT_PLOT)


## 7. Display visual context if available

`ROI_A.gif`, `ROI_B.gif`, etc. are usually the clearest movies for presentations. `movie.gif` is the wider context cutout.


In [ ]:
from IPython.display import Image, Markdown, display

def show_if_exists(path, title):
    path = Path(path)
    if path.exists():
        display(Markdown(f"**{title}**"))
        display(Image(filename=str(path)))

show_if_exists(Path(result_dir) / "lightcurves.png", "Server plot")
show_if_exists(Path(result_dir) / "roi_overlay.png", "ROI overlay image")
for gif in sorted((Path(result_dir) / "roi_movies").glob("*.gif")):
    show_if_exists(gif, f"ROI movie: {gif.name}")
show_if_exists(Path(result_dir) / "movie.gif", "Context movie")


## 8. Export a clean table

This creates a compact CSV for your own analysis. It keeps `UTC`, ROI curves, and `EXPTIME`/`QUALITY` when present.


In [ ]:
EXPORT_CSV = "openmonitor_aia_clean_table.csv"
EXPORT_METADATA = "openmonitor_aia_metadata_summary.json"
DOWNLOAD_OUTPUTS = False  # set True in Colab if you want automatic downloads

keep = ["UTC"] + [col for col in ["EXPTIME", "QUALITY"] if col in df.columns] + roi_columns
clean = df[keep].copy()
clean["UTC"] = clean["UTC"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
clean.to_csv(EXPORT_CSV, index=False)

summary = {
    "source": source,
    "processing": processing,
    "calibration": calibration,
    "quality": quality,
    "pixel_scale": pixel_scale,
    "roi": roi_meta,
    "attribution": attribution,
    "table_columns": list(clean.columns),
}
Path(EXPORT_METADATA).write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("Wrote:", EXPORT_CSV)
print("Wrote:", OUTPUT_PLOT)
print("Wrote:", EXPORT_METADATA)

if DOWNLOAD_OUTPUTS:
    try:
        from google.colab import files
        for name in [EXPORT_CSV, OUTPUT_PLOT, EXPORT_METADATA]:
            files.download(name)
    except Exception as exc:
        print("Automatic download is only available in Colab:", exc)


## Notes for citation and reuse

Use the attribution block printed above when describing the product. The OpenMonitor table is derived from SDO/AIA level-1 cutouts obtained via JSOC. For absolute photometry or cross-date comparison, apply your own AIA calibration/degradation correction and inspect/filter `QUALITY` according to your science case.
